# 08 DreaMS冻结表示 → P/B/T端点：Development五折结构簇交叉验证

本Notebook展示正式结果，核心计算位于 `src/pbt_msms/dreams_heads.py`。DreaMS的1024维embedding完全冻结；模型训练与超参数选择仅使用Development，ChemVista外部测试始终锁定。

主输入为同一化合物全部公共谱图embedding的逐维中位数，因此训练单位是化合物而不是谱图。外层使用冻结Morgan–Butina结构簇5-fold；每个outer训练区内用剩余4个冻结fold选择超参数。预注册主头只在线性模型与两层MLP之间选择；随机森林、CatBoost和SVC/SVR是次要模型。

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'outputs' / 'pbt_msms_v1').is_dir():
    candidates = [parent for parent in Path.cwd().resolve().parents if (parent / 'outputs' / 'pbt_msms_v1').is_dir()]
    if not candidates:
        raise FileNotFoundError('请从项目根目录运行本Notebook。')
    ROOT = candidates[0]
OUTPUT_DIR = ROOT / 'outputs' / 'pbt_msms_v1' / 'dreams_heads_v1'
RUN_FULL_TRAINING = False  # 约40分钟；只有输入或代码改变时才设为True
RUN_FINALIZE_ONLY = False  # 复用外层预测和 embeddings；重拟合内层校准头并复算统计
print('ROOT =', ROOT)
print('Python =', sys.executable)
print('OUTPUT_DIR exists =', OUTPUT_DIR.is_dir())

In [ ]:
if RUN_FULL_TRAINING and RUN_FINALIZE_ONLY:
    raise ValueError('RUN_FULL_TRAINING和RUN_FINALIZE_ONLY不能同时为True。')
if RUN_FULL_TRAINING or RUN_FINALIZE_ONLY:
    environment = os.environ.copy()
    environment['PYTHONPATH'] = str(ROOT / 'src')
    command = [sys.executable, '-m', 'pbt_msms.dreams_heads', '--root', str(ROOT)]
    if RUN_FINALIZE_ONLY:
        command.append('--finalize-existing')
    subprocess.run(command, cwd=ROOT, env=environment, check=True)
else:
    print('使用已经冻结的正式运行结果；未触发重复训练。')

## 1. 运行清单、任务规模与防泄漏QC

In [ ]:
manifest = json.loads((OUTPUT_DIR / 'dreams_head_run_manifest.json').read_text(encoding='utf-8'))
qc = pd.read_csv(OUTPUT_DIR / 'dreams_head_qc_checks.csv')
task_counts = pd.read_csv(OUTPUT_DIR / 'dreams_head_task_counts.csv')
fold_integrity = pd.read_csv(OUTPUT_DIR / 'dreams_head_fold_integrity.csv')
assert manifest['status'] == 'PASS'
assert manifest['external_test_unlocked'] is False
assert qc['status'].eq('PASS').all()
assert fold_integrity['group_overlap'].eq(0).all()
print({
    'run_version': manifest['run_version'],
    'status': manifest['status'],
    'external_test_unlocked': manifest['external_test_unlocked'],
    'qc_passed': f"{qc['status'].eq('PASS').sum()}/{len(qc)}",
    'outer_cv': manifest['outer_cv'],
    'inner_cv': manifest['inner_cv'],
})
display(task_counts)
display(fold_integrity)

## 2. 预注册主结果：线性/两层MLP嵌套选择的化合物级OOF性能

In [ ]:
metrics = pd.read_csv(OUTPUT_DIR / 'dreams_head_metrics.csv')
oof = pd.read_pickle(OUTPUT_DIR / 'dreams_head_compound_level_oof.pkl')
main = metrics[
    metrics['representation_route'].eq('all_spectra_compound_median')
    & metrics['model'].eq('nested_selected_primary')
].copy()
main_columns = [
    'task', 'kind', 'n_compounds', 'primary_metric', 'primary_value',
    'primary_ci_low', 'primary_ci_high', 'mae', 'rmse', 'r2', 'spearman',
    'pr_auc', 'roc_auc', 'mcc', 'balanced_accuracy', 'brier'
]
display(main[main_columns].sort_values('task').round(4))
selected_families = (
    oof[oof['representation_route'].eq('all_spectra_compound_median') & oof['model'].eq('nested_selected_primary')]
    [['task', 'outer_fold', 'selected_family']].drop_duplicates()
    .sort_values(['task', 'outer_fold'])
)
display(selected_families)

## 3. 五类预测头比较

随机森林、CatBoost与SVC/SVR属于预先规定的次要模型。即使次要模型优于主头，也不能在查看outer OOF结果后将其改称预注册主模型。

In [ ]:
family_metrics = metrics[metrics['representation_route'].eq('all_spectra_compound_median')].copy()
family_columns = [
    'task', 'model', 'n_compounds', 'primary_metric', 'primary_value',
    'primary_ci_low', 'primary_ci_high', 'r2', 'spearman', 'brier'
]
display(family_metrics[family_columns].sort_values(['task', 'primary_value']).round(4))

## 4. 多谱图、离子模式与DreaMS DataFormatA适用域敏感性

In [ ]:
sensitivity = metrics[metrics['model'].eq('nested_selected_primary')].copy()
sensitivity_columns = [
    'task', 'representation_route', 'n_compounds', 'primary_metric',
    'primary_value', 'primary_ci_low', 'primary_ci_high', 'r2', 'spearman', 'brier'
]
display(sensitivity[sensitivity_columns].sort_values(['task', 'representation_route']).round(4))

## 5. 与metadata、直接Transformer和结构指纹的配对化合物bootstrap

`observed_delta_positive_is_better > 0` 表示DreaMS更好；95% CI跨0表示没有显著差异。Morgan和MACCS是真实结构上限，不是未知物部署路线。

In [ ]:
paired = pd.read_csv(OUTPUT_DIR / 'dreams_head_paired_bootstrap.csv')
display(paired.sort_values(['task', 'comparator']).round(4))

## 6. P概率校准与最终QC

非 SVM 的 P 概率采用严格的 outer-train inner-OOF Platt 校准。每个外层折内，在其余四折重新构造结构分组的内层 OOF 分数；产生校准分数的预测头和 Platt 拟合均不使用当前外层测试身份。选定超参数仅来源于当前外层训练集。SVC 沿用外层训练区内的结构分组校准。原始外层概率保存在 `y_probability_raw`。

本节报告修正后的内部验证。既有外部验证、解释性与盲态现场部署的模型及参数另存 `frozen_deployment_v1`，不使用外部标签重新选取校准方案。


In [ ]:
calibration = pd.read_csv(OUTPUT_DIR / 'dreams_head_probability_calibration.csv')
audit = pd.read_csv(OUTPUT_DIR / 'dreams_head_calibration_fit_audit.csv')
assert calibration['current_fold_excluded'].all()
assert calibration['base_fit_outer_fold_excluded'].all()
assert audit['outer_test_overlap_n'].eq(0).all()
assert audit['inner_group_overlap_n'].eq(0).all()
print('Calibration rows =', len(calibration))
print('Audited inner fits =', len(audit))
print('Raw probability retained =', 'y_probability_raw' in oof.columns)
display(calibration.head(10))
display(audit.head(10))
display(qc)


## 7. 当前结论边界

- DreaMS冻结表示对P筛查分类提供了明确有效信息；主头显著优于metadata与当前Set Transformer。
- 预注册主回归头仅LC50明确优于dummy；logBCF和EC50未显著优于dummy，NOEC略差于dummy。不能把这些回归端点描述成已经获得稳定预测。
- SVC/SVR次要模型普遍优于线性/MLP，说明DreaMS空间中的非线性核可能更适合小样本端点；这一发现应作为次要/探索性结果，在外部测试前冻结后续规则。
- 所有DreaMS路线仍弱于Morgan/MACCS结构上限，符合结构指纹作为性能上限参照的定位。
- ChemVista尚未解锁；本Notebook只回答Development内部结构外推能力，不能替代独立实验谱图验证。